# MMLU Multilingual Benchmark (Google Colab)

This notebook removes Kaggle/KaggleHub dependencies. Upload the required development and test Parquet files directly to Colab, then run the benchmark with the Gemini API.

**Required files for English:**  
- `Global-MMLU-Lite-en-dev.parquet`
- `Global-MMLU-Lite-en-test.parquet`

For other languages, upload matching files such as `Global-MMLU-Lite-es-dev.parquet` and `Global-MMLU-Lite-es-test.parquet`.

The Parquet files must contain: `subject`, `question`, `option_a`, `option_b`, `option_c`, `option_d`, `answer`. `cultural_sensitivity_label` is optional.

## 1. Install and import libraries

In [ ]:
!pip -q install pandas pyarrow requests openpyxl

In [ ]:
import re
import json
import time
import pandas as pd
import requests
from pathlib import Path
from google.colab import files, userdata

print("Libraries imported successfully.")

## 2. Add your Gemini API key

In Colab, open the **Secrets** panel (key icon on the left), add a secret named `GOOGLE_AI_API_KEY`, paste your Gemini API key as its value, and enable notebook access.

Do not paste your API key into a public notebook or share it with anyone.

In [ ]:
try:
    GOOGLE_AI_API_KEY = userdata.get("GOOGLE_AI_API_KEY")
except Exception:
    GOOGLE_AI_API_KEY = None

if not GOOGLE_AI_API_KEY:
    raise ValueError(
        "GOOGLE_AI_API_KEY was not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

# You can change the model name here if your Google AI account uses another model.
MODEL_NAME = "gemini-2.5-flash"

print("Gemini API key loaded. Key value is hidden.")

## 3. Upload your Parquet dataset files

In [ ]:
uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print("-", filename)

If your dataset files are already in Google Drive instead, mount Drive and set `DATA_DIR` to their folder. For this upload-based notebook, the files are stored in `/content`.

In [ ]:
DATA_DIR = Path("/content")

def load_language_data(language="en"):
    dev_path = DATA_DIR / f"Global-MMLU-Lite-{language}-dev.parquet"
    test_path = DATA_DIR / f"Global-MMLU-Lite-{language}-test.parquet"

    missing = [str(p) for p in [dev_path, test_path] if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing dataset file(s): " + ", ".join(missing) +
            ". Upload both development and test Parquet files."
        )

    dev_df = pd.read_parquet(dev_path)
    test_df = pd.read_parquet(test_path)

    required_columns = {
        "subject", "question", "option_a", "option_b",
        "option_c", "option_d", "answer"
    }

    for label, df in [("development", dev_df), ("test", test_df)]:
        missing_columns = required_columns - set(df.columns)
        if missing_columns:
            raise ValueError(
                f"{label} dataset is missing columns: {sorted(missing_columns)}"
            )

    return dev_df, test_df

print("Dataset loader is ready.")

## 4. Connect to Gemini and send prompts

In [ ]:
def prompt_gemini(prompt_text, temperature=0.0, max_retries=3):
    url = (
        f"https://generativelanguage.googleapis.com/v1beta/models/"
        f"{MODEL_NAME}:generateContent"
    )

    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": GOOGLE_AI_API_KEY
    }

    payload = {
        "contents": [
            {"role": "user", "parts": [{"text": prompt_text}]}
        ],
        "generationConfig": {
            "temperature": temperature,
            "maxOutputTokens": 256
        }
    }

    for attempt in range(max_retries):
        response = requests.post(
            url, headers=headers, json=payload, timeout=60
        )

        if response.status_code == 200:
            data = response.json()
            try:
                return data["candidates"][0]["content"]["parts"][0]["text"]
            except (KeyError, IndexError, TypeError):
                raise RuntimeError(f"Unexpected Gemini response: {data}")

        if response.status_code in (429, 500, 502, 503) and attempt < max_retries - 1:
            time.sleep(2 ** attempt)
            continue

        raise RuntimeError(
            f"Gemini API error {response.status_code}: {response.text}"
        )

    raise RuntimeError("Gemini API request failed after retries.")

## 5. Prepare prompts and extract answers

In [ ]:
def normalize_answer(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    match = re.search(r"\b([A-D])\b", value)
    return match.group(1) if match else value


def prepare_user_prompt(row, dev_df, examples_per_subject=2):
    subject = str(row["subject"])
    subject_df = dev_df[dev_df["subject"].astype(str) == subject].head(examples_per_subject)

    prompt_parts = [
        f"You are answering a multiple-choice question in the subject: {subject}.",
        "Use the examples as guidance. Select exactly one option.",
        "Return your final answer in the format: Answer: A",
        ""
    ]

    for _, example in subject_df.iterrows():
        prompt_parts.append(f"Question: {example['question']}")
        prompt_parts.append(f"A. {example['option_a']}")
        prompt_parts.append(f"B. {example['option_b']}")
        prompt_parts.append(f"C. {example['option_c']}")
        prompt_parts.append(f"D. {example['option_d']}")
        prompt_parts.append(f"Answer: {normalize_answer(example['answer'])}")
        prompt_parts.append("")

    prompt_parts.extend([
        "Now answer this question:",
        f"Question: {row['question']}",
        f"A. {row['option_a']}",
        f"B. {row['option_b']}",
        f"C. {row['option_c']}",
        f"D. {row['option_d']}",
        "Answer:"
    ])

    return "\n".join(prompt_parts)


def extract_answer(response):
    text = str(response).strip().upper()

    # Prefer explicit answer labels.
    patterns = [
        r"ANSWER\s*[:：-]\s*\(?([A-D])\)?",
        r"OPTION\s*[:：-]\s*\(?([A-D])\)?",
        r"^\s*\(?([A-D])\)?[.\s]*$"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.MULTILINE)
        if match:
            return match.group(1)

    # Fallback: a standalone A/B/C/D at the beginning of a line.
    match = re.search(r"^\s*\(?([A-D])\)?[.)]?(?:\s|$)", text, re.MULTILINE)
    return match.group(1) if match else "Unknown"

## 6. Run one question and calculate accuracy

In [ ]:
def mmlu_run_task(row, dev_df):
    prompt_text = prepare_user_prompt(row, dev_df)
    response_text = prompt_gemini(prompt_text, temperature=0.0)

    predicted = extract_answer(response_text)
    correct_answer = normalize_answer(row["answer"])
    label = row.get("cultural_sensitivity_label", "Unknown")

    return {
        "subject": row["subject"],
        "question": row["question"],
        "response": response_text,
        "predicted_answer": predicted,
        "correct_answer": correct_answer,
        "correct": predicted == correct_answer,
        "cultural_sensitivity_label": label
    }


def calculate_accuracy(results, sensitivity=None):
    if sensitivity is not None:
        results = [
            item for item in results
            if str(item.get("cultural_sensitivity_label", "")).upper() == sensitivity.upper()
        ]

    if not results:
        return 0.0

    return sum(bool(item["correct"]) for item in results) / len(results)


print("Benchmark functions are ready.")

## 7. Run the English benchmark

In [ ]:
LANGUAGE = "en"
SAMPLE_SIZE = 10  # Increase this after confirming the first few examples work.

dev_df, test_df = load_language_data(LANGUAGE)

print(f"Loaded {len(dev_df)} development rows and {len(test_df)} test rows.")
display(test_df.head(3))

results = []

for position, (_, row) in enumerate(test_df.head(SAMPLE_SIZE).iterrows(), start=1):
    result = mmlu_run_task(row, dev_df)
    results.append(result)

    print(
        f"Example {position}: "
        f"predicted={result['predicted_answer']}, "
        f"correct={result['correct_answer']}, "
        f"match={result['correct']}"
    )

overall_accuracy = calculate_accuracy(results)
cs_accuracy = calculate_accuracy(results, "CS")
ca_accuracy = calculate_accuracy(results, "CA")

print(f"\nOverall accuracy: {overall_accuracy:.2%}")
print(f"CS accuracy: {cs_accuracy:.2%}")
print(f"CA accuracy: {ca_accuracy:.2%}")

## 8. Optional multilingual benchmark

In [ ]:
def run_multilingual_benchmark(languages=("en", "es", "fr", "de", "zh"), sample_size=5):
    summary_rows = []
    all_language_results = {}

    for language in languages:
        print(f"\n=== Language: {language} ===")
        try:
            dev_data, test_data = load_language_data(language)
            language_results = []

            for _, row in test_data.head(sample_size).iterrows():
                language_results.append(mmlu_run_task(row, dev_data))

            accuracy = calculate_accuracy(language_results)
            cs_acc = calculate_accuracy(language_results, "CS")
            ca_acc = calculate_accuracy(language_results, "CA")

            summary_rows.append({
                "language": language,
                "sample_count": len(language_results),
                "accuracy": accuracy,
                "CS_accuracy": cs_acc,
                "CA_accuracy": ca_acc
            })

            all_language_results[language] = language_results
            print(f"Accuracy: {accuracy:.2%}")

        except FileNotFoundError as error:
            print(f"Skipping {language}: {error}")
        except Exception as error:
            print(f"Error for {language}: {error}")

    return pd.DataFrame(summary_rows), all_language_results


# Run only for languages whose dev and test Parquet files you uploaded.
# Example:
# multilingual_summary, multilingual_results = run_multilingual_benchmark(
#     languages=("en", "es", "fr"), sample_size=5
# )
# display(multilingual_summary)

## 9. Save benchmark results

In [ ]:
results_df = pd.DataFrame(results)

summary_df = pd.DataFrame([{
    "language": LANGUAGE,
    "sample_count": len(results),
    "overall_accuracy": calculate_accuracy(results),
    "CS_accuracy": calculate_accuracy(results, "CS"),
    "CA_accuracy": calculate_accuracy(results, "CA")
}])

output_name = "MMLU_Benchmark_Results.xlsx"

with pd.ExcelWriter(output_name, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Detailed Results", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"Saved: {output_name}")
files.download(output_name)

## Notes

- This notebook expects the same MMLU Lite Parquet dataset structure as your original script, but it no longer downloads anything from Kaggle.
- You must obtain the dataset files separately and upload the matching `dev` and `test` Parquet files.
- The multilingual benchmark only runs for languages whose two Parquet files are uploaded.
- API usage may incur charges or be subject to account quotas. Start with a small `SAMPLE_SIZE`.
- The original script's `translated_prompts_final.json` file was not actually required by its prompt-building function, so this corrected notebook removes that unused dependency.